In [ ]:
# 1. Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from datetime import datetime
from collections import Counter
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

nltk.download('stopwords')

# Notebook settings
%matplotlib inline
sns.set(style='whitegrid')


In [ ]:
# 2. Load analyst ratings data
analyst_df = pd.read_csv("data/raw/raw_analyst_ratings.csv", parse_dates=["date_published"])

print("Dataset shape:", analyst_df.shape)
analyst_df.head()


In [ ]:
# 3. Descriptive Statistics
analyst_df['headline_length'] = analyst_df['headline'].apply(lambda x: len(str(x).split()))

# Basic stats on headline length
print("Headline Length Stats:")
print(analyst_df['headline_length'].describe())

# Articles per publisher
publisher_counts = analyst_df['publisher'].value_counts()
plt.figure(figsize=(10,5))
publisher_counts.head(10).plot(kind='bar')
plt.title('Top 10 Publishers by Article Count')
plt.ylabel('Count')
plt.xlabel('Publisher')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Articles over time
analyst_df['date_only'] = analyst_df['date_published'].dt.date
date_counts = analyst_df['date_only'].value_counts().sort_index()

plt.figure(figsize=(12,4))
date_counts.plot()
plt.title('Article Publication Over Time')
plt.ylabel('Number of Articles')
plt.xlabel('Date')
plt.tight_layout()
plt.show()


In [ ]:
# 4. Text Analysis - Topic Modeling (LDA)
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    tokens = [word for word in text.split() if word not in stop_words and len(word) > 2]
    return " ".join(tokens)

analyst_df['cleaned_headline'] = analyst_df['headline'].apply(preprocess)

# Vectorize
vectorizer = CountVectorizer(max_df=0.9, min_df=10)
doc_term_matrix = vectorizer.fit_transform(analyst_df['cleaned_headline'])

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(doc_term_matrix)

# Display topics
words = vectorizer.get_feature_names_out()
for idx, topic in enumerate(lda.components_):
    print(f"Topic {idx + 1}:")
    print([words[i] for i in topic.argsort()[-10:]])
    print()


In [ ]:
# 5. Time-of-day analysis
analyst_df['hour'] = analyst_df['date_published'].dt.hour

plt.figure(figsize=(10,4))
analyst_df['hour'].value_counts().sort_index().plot(kind='bar')
plt.title("Articles Published by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Article Count")
plt.tight_layout()
plt.show()


In [ ]:
# 6. Publisher Email/Domain Analysis
if analyst_df['publisher'].str.contains("@").any():
    analyst_df['domain'] = analyst_df['publisher'].str.extract(r'@([\w\.-]+)')
    domain_counts = analyst_df['domain'].value_counts()

    plt.figure(figsize=(8,4))
    domain_counts.head(10).plot(kind='bar')
    plt.title("Top 10 Publisher Email Domains")
    plt.ylabel("Count")
    plt.xlabel("Domain")
    plt.tight_layout()
    plt.show()
else:
    print("No email-style publishers found.")


In [ ]:
# 7. Financial data load example
stock_data = {}
base_path = "data/raw/yfinance_data"

for filename in os.listdir(base_path):
    if filename.endswith(".csv"):
        ticker = filename.split("_")[0]
        df = pd.read_csv(os.path.join(base_path, filename), parse_dates=["Date"])
        stock_data[ticker] = df
        print(f"{ticker}: {df.shape}")

# Preview AAPL data
stock_data['AAPL'].head()
